In [229]:
from FoKL import FoKLRoutines
import pyomo.environ as pyo
import numpy as np
import pyomo.dae as dae

Example GP:

In [230]:
t = np.linspace(0, 99, 100)
x1 = t ** 2
x2 = np.sin(t)
y = x1 + x2

try:
    GP = FoKLRoutines.load('test_model.fokl')
except Exception as exception:
    GP = FoKLRoutines.FoKL(kernel=1, UserWarnings=False)
    _ = GP.fit([x1, x2], y, clean=True)
    GP.save('test_model.fokl')

User inputs:

In [231]:
xvar = ['x1', 'x2']
yvar = 'y'
m_global = pyo.ConcreteModel('test model')
draws = 5
ode = True
t_span = [2.4, 9.1]
name = 'gp name test'

mtx = GP.mtx
betas = GP.betas
minmax = GP.minmax
phis = GP.phis

## Create sub-model ```m``` of GP only, prior to merging with ```m_global```

- ```xvar``` and ```yvar``` defined in ```m_global```

In [232]:
m = pyo.ConcreteModel(name)

# Some constants:
mtx = np.array(mtx, dtype=int)  # indices/orders of basis functions (where 1 is B1 and 0 means none)

# Some sets:
m.draws = pyo.Set(initialize = range(draws))
m.terms = pyo.Set(initialize = range(mtx.shape[0] + 1))  # terms (including beta0)
m.orders = pyo.Set(initialize = np.unique(mtx[mtx != 0]))  # orders of basis functions
m.attributes = pyo.Set(initialize = range(mtx.shape[1]))  # input variables

## Define betas variables

In [233]:
m.beta = pyo.Var(m.draws, m.terms, within=pyo.Reals)

m.beta_avg = pyo.Var(m.terms, within=pyo.Reals)

In [234]:
def fix_betas(m, betas):
    """Fix the already-initialized Pyomo beta variables to scalar values in 'betas', using last 'betas' draw as first Pyomo draw."""
    for draw in m.draws:
        for term in m.terms:
            m.beta[draw, term].fix(betas[-(draw + 1), term])

## Define time for ODE

In [235]:
if ode is True:
    m.t = dae.ContinuousSet(bounds=t_span)
else:
    m.t = pyo.Set(initialize=range(1))  # single index to avoid if-else statements in internal code

## Define normalized input variables (attributes)

In [236]:
m.x = pyo.Var(m.t, m.attributes, within=pyo.Reals, bounds=(0, 1))

## Define phi "basis" functions

- which are not technically "bases" because Bernoulli's are not sets of vectors in same vector space (each "basis" function for Bernoulli is single "vector" in its own vector space, whereas Cubic Splines are 500 sets of 499 4D vectors)
    - hence, using ```orders``` not ```bases```

Pyomo expression as callable function:
- https://groups.google.com/g/pyomo-forum/c/LJkkyHxZT1A/m/NBa7W9TWL-kJ

Example of syntax with ContinuousSet:
- https://pyomo.readthedocs.io/en/stable/modeling_extensions/dae.html#using-the-simulator

In [237]:
nj = []  # list of [order, attribute] combinations used in GP
for attribute in m.attributes:
    orders_j = np.unique(mtx[:, attribute])
    if any(orders_j != 0):
        for order_j in orders_j[orders_j != 0]:
            nj.append([order_j, attribute])

In [238]:
def _rule(m, t, n, j):
    nm1 = n - 1  # Python indexing, since n=1 refers to B1 which is phis[0]
    return phis[nm1][0] + sum(phis[nm1][k] * m.x[t, j] ** k for k in range(1, len(phis[nm1])))

m.phi = pyo.Expression(m.t, nj, rule=_rule)

## Build GP expression

In [239]:
m.y = pyo.Var(m.t, m.draws, within=pyo.Reals)

def _rule2(m, t, draw):
    y = m.beta[draw, 0]  # FoKL's GP equation; initialization
    
    for term in range(1, len(m.terms)):  # == m.terms[1::]
        y_term = m.beta[draw, term]

        for j in m.attributes:
            n = mtx[term - 1, j]

            if n != 0:  # since 0 means none
                y_term *= m.phi[t, n, j]

        y += y_term

    return m.y[t, draw] == y

m.gp = pyo.Constraint(m.t, m.draws, rule=_rule2)

In [240]:
m.gp.pprint()

gp : Size=10, Index=t*draws, Active=True
    Key      : Lower : Body                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             : Upper : Active
    (2.4, 0) :   0.0 : y[2.4,0] - (beta[0,0] + beta[0,1]*(-0.5196152752436607 + 1.0392305504873183*x[2.4,1]) + beta[0,2]*(-0.5196152752436607 + 1.0392305504873183*x[2.4,0]) + beta[0,3]*(-1.0912508854411944*x[2.4,1] + 1.0912508854411944*x[2.4,1]**2 + 0.18187514757353293) + beta[0,4]*(-1.0912508854411944*x[2.4,0] + 1.0912508854411944*x[2.4,0]**2 + 0.18187514757353293)*(-0.5196152752436607 